In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from IPython.display import display, Markdown

mpl.rcParams['font.size'] = 16

In [2]:
def read_file(filename):
    with open(filename) as f:
        data = []
        for line in f:
            line =  line.rstrip().lstrip()
            if not line:
                continue

            temp = line.split()
            data.append(float(temp[4]))

    return data

def read_file_with_smiles(filename):
    with open(filename) as f:
        data = []
        for line in f:
            line =  line.rstrip().lstrip()
            if not line:
                continue

            temp = line.split()
            data.append(float(temp[5]))

    return data

def plot_hist(data, output):
    fig = plt.figure(figsize=(10, 5))
    ax1 = fig.add_subplot(111)
    ax1.hist(data, bins=range(0, int(np.max(data))))
    ax1.set_xlabel('time [ms]')
    #ax1.set_xlim(0, 90)
    ax1.set_ylabel('Queries #')

    fig.savefig(f'{output}-hist.png', dpi=300)
    plt.show()

def percentiles(filename):
    data = read_file(filename)
    return np.percentile(data, 99), np.percentile(data, 90), np.percentile(data, 75), np.percentile(data, 50), np.percentile(data, 25)

def plots(filename, title, smiles=False):
    if smiles:
        data = read_file_with_smiles(filename)
    else:
        data = read_file(filename)
    md_output=f'''
## {title}
**Total time**: {np.sum(data):.3f} ms

**Average time**: {np.mean(data):.3f} ms

| Queries resolved (percentile)  | Time (ms) |
| -------------------------------| -----------|
| 99  | {np.percentile(data, 99):.3f} |
| 90  | {np.percentile(data, 90):.3f} |
| 75  | {np.percentile(data, 75):.3f} |
| 50  | {np.percentile(data, 50):.3f} |
| 25  | {np.percentile(data, 25):.3f} |
    '''
    display(Markdown(md_output))
    plot_hist(data, title)

In [3]:
percentile_data = {}
cutoffs =  {"10": 0.10, "15": 0.15, "10": 0.10, "20": 0.2, "22": 0.22, "24": 0.24, "26": 0.26, "28": 0.28, "30": 0.30,
            "32": 0.32, "34": 0.34, "36": 0.36, "38": 0.38, "40": 0.40, "42": 0.42, "44": 0.44}

percentile_data["FPSim2"] = percentiles(f"../fpsim2_benchmark/benchmarking_b256_0_7.log")
for key, value in cutoffs.items():
    percentile_data[str(value)] = percentiles(f"benchmark_{key}.log")

In [4]:
header = ["cut-off", "99%", "90%", "75%", "50%", "25%"]
with open("benchmark_b256.csv", "w") as fin:
    fin.write(",".join(header))
    fin.write("\n")
    for key, value in percentile_data.items():
        fin.write(f"{key},")
        fin.write(",".join(map(lambda x: f'{x:.2f}',value)))
        fin.write("\n")
